In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
#canonical_cards.json
import json

canonical_cards = [
  {
    "card_id": "DENGUE_DIAG_01",
    "disease": "Dengue",
    "clinical_category": "probable_diagnosis",
    "prerequisites": [
      "Acute onset of high-grade continuous fever (39-40 C / 102-104 F) lasting 2-7 days",
      "Two or more of: nausea/vomiting, macular or maculopapular rash, severe headache, retro-orbital pain, myalgia/arthralgia (break-bone fever), positive tourniquet test, or leucopenia (WBC < 5000/mm3)"
    ],
    "recommended_action": "Classify as Suspected/Probable Dengue Fever. Evaluate immediately for Dengue Warning Signs (intense abdominal pain, persistent vomiting >=3 episodes/1 hr or >=4/6 hrs, mucosal bleeding, lethargy/irritability, hepatomegaly >=2cm, fluid accumulation, or progressive haematocrit rise) and high-risk comorbidities to stratify into Dengue Without Warning Signs (outpatient/home care) vs. Dengue With Warning Signs or High-Risk (inpatient admission).",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Must check for warning signs (page 28) and hemodynamic stability (BP, pulse, urine output)",
      "Must check high-risk conditions requiring admission even without warning signs: pregnancy, age <1 yr or >65 yrs, obesity, diabetes, hypertension, chronic renal/liver/heart disease, peptic ulcer, hemolytic disease, steroid/NSAID/anticoagulant use, or living alone/poor healthcare access",
      "Consider differential diagnosis: Chikungunya (prominent severe symmetric polyarthralgia/arthritis), Zika, Malaria, Enteric fever"
    ],
    "source": {
      "title": "National Guideline for Clinical Management of Dengue",
      "version": "Fifth Edition",
      "pdf_page": "26-28, 45-47",
      "evidence_passage": "Classical dengue fever characterized by high fever, severe headache, retro-orbital pain, myalgia, arthralgia and rashes. Leucopenia and thrombocytopenia are usually observed... Dengue without warning signs: Fever (2-7 days) with two or more of the followings; nausea, vomiting, rash, aches and pains, positive tourniquet tests, or leucopenia (WBC < 5000/mm3). Dengue with warning signs: Dengue fever with any one of the warning signs."
    }
  },
  {
    "card_id": "DENGUE_INV_01",
    "disease": "Dengue",
    "clinical_category": "investigations",
    "prerequisites": [
      "Suspected or probable Dengue fever",
      "Duration of fever known (<= 5 days vs. > 5 days from onset)"
    ],
    "recommended_action": "If presenting within 5 days of fever onset: order baseline Complete Blood Count (CBC) with Haematocrit (Hct) and platelet count, Dengue NS1 Antigen test (or RT-PCR), and liver transaminases (SGOT/AST, SGPT/ALT). If presenting after 5 days of fever (Day 6 onward): order CBC with Hct and Dengue IgM/IgG Antibody test (since NS1 may become negative from day 4-5). Review daily with CBC starting at least on the 3rd day of illness until afebrile for 24-48 hours.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Missing information to check: Exact day of fever onset is mandatory to choose between NS1 Antigen (days 1-5) and Dengue IgM/IgG antibody (day 6+)",
      "Note: Rising Hct >10% indicates early plasma leakage; >=20% rise from baseline (assumed 35% child, 38% adult female, 40% adult male) indicates established plasma leakage (DHF). Platelet <100,000/mm3 and NLR <2 precede/accompany leakage."
    ],
    "source": {
      "title": "National Guideline for Clinical Management of Dengue",
      "version": "Fifth Edition",
      "pdf_page": "37-39, 41, 45, 48",
      "evidence_passage": "Basic investigations depend on day of illness. CBC and NS1 Ag up to day 5 and CBC and Dengue Antibody from day 6 onward will support the diagnosis... Within 5 days: CBC, Haematocrit, Dengue NS1 antigen, SGOT, SGPT. These tests should be done during first consultation to get the baseline characteristics... Review daily with CBC."
    }
  },
  {
    "card_id": "DENGUE_MED_01",
    "disease": "Dengue",
    "clinical_category": "medication_guidance",
    "prerequisites": [
      "Suspected or confirmed Dengue fever suitable for outpatient/home management (no warning signs, normal vital signs and urine output, platelet >= 100x10^9/L, no high-risk comorbidities)"
    ],
    "recommended_action": "Prescribe oral Paracetamol for high fever and pain along with tepid sponging, and oral rehydration fluids with electrolytes (ORS, green coconut water, fruit juice, or soup) to maintain hydration. Anti-emetics and PPI/H2 receptor blockers may be given if necessary for nausea/gastritis. Strictly avoid all NSAIDs and steroids.",
    "medication_details": [
      {
        "name": "Paracetamol",
        "dose": "Max 3 gm/day in adults (60 mg/kg/24 hrs in children)",
        "route": "Oral",
        "frequency": "Divided doses as needed for fever",
        "duration": "During febrile period",
        "instructions": "Do not exceed 3 gm/day in adults. Warn patient that fever may not fully settle with paracetamol and not to take excess."
      },
      {
        "name": "Oral Rehydration Fluid (ORS / Electrolyte fluids)",
        "dose": "2400 mL per 24 hours for adults >=50 kg (or 2 mL/kg/hour for 24 hours if body weight <50 kg)",
        "route": "Oral",
        "frequency": "Spread evenly across the 24-hour day",
        "duration": "Throughout acute febrile and defervescence phase",
        "instructions": "Use ORS, green coconut water, fruit juices, or soup containing electrolytes rather than plain water alone."
      }
    ],
    "exclusions_or_missing_info": [
      "STRICT EXCLUSION: Avoid all NSAIDs (including COX-2 inhibitors), aspirin, and steroids due to severe bleeding risk",
      "Temporarily withhold long-term warfarin, aspirin, clopidogrel, and dipyridamole (seek expert opinion if essential)",
      "Missing information to check: Patient body weight (to adjust fluid to 2 mL/kg/hr if <50 kg or pediatric dose 60 mg/kg/day), current medications (anticoagulants/antiplatelets/NSAIDs), and liver disease history"
    ],
    "source": {
      "title": "National Guideline for Clinical Management of Dengue",
      "version": "Fifth Edition",
      "pdf_page": "47-48",
      "evidence_passage": "Do not exceed 3 gm/day of paracetamol. (60mg/kg/24hrs in child)... Ensure adequate oral fluid intake of around 2400 ml for 24 hours (if the body weight is less than 50kg, give fluids as 2ml/kg/hour for 24 hours)... Avoid all NSAIDS (including COX-2 inhibitors) and steroids. Temporally withhold warfarin, aspirin, clopidogrel & dipyridamole."
    }
  },
  {
    "card_id": "DENGUE_ADV_01",
    "disease": "Dengue",
    "clinical_category": "advice",
    "prerequisites": [
      "Suspected or probable Dengue fever managed at home or outpatient department"
    ],
    "recommended_action": "Advise adequate physical rest under a mosquito net, tepid sponging, light nutritious diet (avoid dark red/brown foods like beet root or red dragon fruit that mimic blood in vomit/stool), and passing urine every 3-4 hours. Follow up daily with CBC until afebrile for 24-48 hours. Instruct patient to return to hospital IMMEDIATELY if any warning sign develops, especially during defervescence (settling of fever).",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Immediate hospitalization required if any red flag appears: clinical deterioration as fever settles, inability to tolerate oral fluids, severe abdominal pain, persistent vomiting, lethargy/restlessness, mucosal bleeding (nose, gums, vaginal, black tarry stool, blood vomit), reduced urine output (<150-200 mL in 4-6 hours for >=50 kg), cold/clammy extremities, or difficulty breathing"
    ],
    "source": {
      "title": "National Guideline for Clinical Management of Dengue",
      "version": "Fifth Edition",
      "pdf_page": "28, 47-48",
      "evidence_passage": "Settling of fever is not a sign of recovery. Some patients will start plasma leakage around defervescence... It is advisable to avoid red/brown (i.e. dark) colored food... Advise patients to return to the hospital immediately, if they develop any of the following conditions: Clinical deterioration with settling of fever, Inability to tolerate oral fluids, Severe abdominal pain, Lethargy or irritability/restlessness, Bleeding tendency... Reduced urine output for the last 4-6 hours."
    }
  },
  {
    "card_id": "DIFF_DENGUE_CHIK_01",
    "disease": "Dengue / Chikungunya",
    "clinical_category": "probable_diagnosis",
    "prerequisites": [
      "Acute febrile illness with headache, rash, myalgia, or arthralgia where clinical features overlap between Dengue and Chikungunya"
    ],
    "recommended_action": "Consider both Dengue and Chikungunya in the differential diagnosis. Differentiate using high-certainty clinical/lab markers: Chikungunya is strongly favored by severe symmetric polyarthralgia/arthritis (++++ vs +/- in Dengue, often wrists/ankles/hands with morning stiffness and stooping posture) and lymphopenia, whereas Dengue is differentiated by thrombocytopenia (+++), progressive increase in haematocrit (hemoconcentration), neutropenia/leucopenia, retro-orbital pain, vomiting, and abdominal pain. CRITICAL POLICY: Manage all suspected Chikungunya patients as Dengue (avoiding aspirin/NSAIDs and monitoring CBC/Hct) until Dengue has been ruled out.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Missing information to check: Duration of fever (for NS1/RT-PCR vs IgM), presence of symmetric joint swelling/arthritis vs bleeding/warning signs, baseline CBC (platelet count and haematocrit), and pregnancy/comorbidity status",
      "Do not prescribe NSAIDs while diagnosis is ambiguous because NSAIDs increase hemorrhage risk if the illness is Dengue"
    ],
    "source": {
      "title": "National Guideline for Clinical Management of Dengue (5th Ed) & National Guideline on Chikungunya Management (2017)",
      "version": "Fifth Edition / 2017",
      "pdf_page": "Dengue Page 29; Chikungunya Pages 16-18",
      "evidence_passage": "Difficult to distinguish chikungunya and dengue based on clinical findings alone... Chikungunya virus more likely to cause high fever, severe polyarthralgia, arthritis, rash, and lymphopenia. Dengue virus more likely to cause neutropenia, thrombocytopenia, hemorrhage, shock, and death. Patients with suspected chikungunya should be managed as dengue until dengue has been ruled out. Aspirin and other NSAIDs can increase the risk of hemorrhage in patients with dengue."
    }
  },
  {
    "card_id": "HTN_DIAG_01",
    "disease": "Hypertension",
    "clinical_category": "probable_diagnosis",
    "prerequisites": [
      "Adult >= 18 years old",
      "Elevated office blood pressure measurement (SBP >= 140 mmHg and/or DBP >= 90 mmHg)"
    ],
    "recommended_action": "If office BP is >= 160/100 mmHg OR accompanied by Hypertension-Mediated Organ Damage (HMOD, e.g., LVH, retinopathy, renal damage), confirm Hypertension on a single visit and evaluate/initiate drug treatment immediately. If initial office BP is 140-159/90-99 mmHg without HMOD, classify as Probable Grade 1 Hypertension, advise lifestyle modification, and confirm diagnosis with repeated measurements (at least 3 readings per visit across 2-3 visits at 1-4 week intervals, or after 2 weeks per primary care protocol, or via out-of-office HBPM/ABPM).",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Initial evaluation must measure BP in both arms (if difference >10 mmHg persists, use higher arm; if >20 mmHg, investigate further)",
      "In elderly patients, diabetics, or those with dizziness/falls, measure lying BP (5 mins) and standing BP (at 1 and 3 mins) to rule out postural hypotension (SBP drop >=20 mmHg or DBP drop >=10 mmHg)",
      "Missing information to check: Number of prior BP visits/readings, presence of HMOD or CVD symptoms (chest pain, breathlessness, visual changes, neurological deficits, edema), diabetes/CKD history, and pregnancy status"
    ],
    "source": {
      "title": "Bangladesh Guidelines on Hypertension",
      "version": "Second Edition 2023",
      "pdf_page": "46-47, 64-65",
      "evidence_passage": "The diagnosis might be made on a single visit, if BP is >=160/100 mmHg or there is accompanying evidence of hypertension mediated organ damage (HMOD)... When the initial SBP is between 140 and 159 mmHg, or the DBP is between 90 and 99 mmHg, repeat measurements should be performed on three separate occasions within a period of 1 to 4 weeks... Primary Care Protocol (Page 64): If SBP 140-159 or DBP 90-99 mmHg -> Advice for lifestyle modification -> Review BP after 2 weeks."
    }
  },
  {
    "card_id": "HTN_INV_01",
    "disease": "Hypertension",
    "clinical_category": "investigations",
    "prerequisites": [
      "Suspected or confirmed Hypertension in an adult"
    ],
    "recommended_action": "Order routine baseline investigations for all hypertensive patients to screen for comorbidities and HMOD: Blood tests for serum sodium, potassium, serum creatinine, and estimated GFR (eGFR); lipid profile and fasting blood glucose (if available); Urine dipstick analysis (and urine albumin:creatinine ratio if renal disease suspected); and a 12-lead ECG (to detect atrial fibrillation, LVH, or ischemic heart disease). Before initiating and several weeks after starting Losartan/ACEi/ARB or diuretics, check serum creatinine and potassium, and perform a pregnancy test in women of childbearing age.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "CRITICAL TIMING RULE: Do not delay starting pharmacological treatment while waiting for laboratory tests or ECG, especially in low-resource settings or when BP >= 160/100 mmHg. Long-acting dihydropyridine CCBs (Amlodipine) can be initiated safely without prior lab testing.",
      "Additional targeted tests if HMOD or secondary cause suspected: Fundoscopy (Grade 2/3 HTN), Echocardiogram, Chest X-ray"
    ],
    "source": {
      "title": "Bangladesh Guidelines on Hypertension",
      "version": "Second Edition 2023",
      "pdf_page": "51-52, 64",
      "evidence_passage": "All hypertensive patients should undergo a limited number of investigations: Blood test (Sodium, potassium, serum creatinine and eGFR; if available lipid profile and fasting glucose), Urine analysis: Dipstick urine test, 12-lead ECG... In low-resourced areas... treatment should not be delayed, and testing can be done subsequently. Page 64 footnote c: Before initiating (and several weeks after starting) losartan, check serum creatinine and serum potassium. Avoid losartan for women of childbearing age who may become pregnant."
    }
  },
  {
    "card_id": "HTN_MED_01",
    "disease": "Hypertension",
    "clinical_category": "medication_guidance",
    "prerequisites": [
      "Adult >= 18 years with confirmed Hypertension",
      "NOT pregnant and NOT a woman of childbearing age who may become pregnant",
      "Persistent BP 140-159/90-99 mmHg after 2 weeks of lifestyle modification OR initial BP >= 160/100 mmHg"
    ],
    "recommended_action": "Follow the Primary Health Care Stepped Protocol: For persistent BP 140-159/90-99 mmHg, start Step 1 (Amlodipine 5 mg once daily; use 2.5 mg in elderly or low body weight). For initial BP >= 160/100 mmHg, start immediate treatment at Step 2 (Amlodipine 5 mg + Losartan 50 mg once daily). Review monthly: if target BP (<140/90 mmHg for most patients; <130/80 mmHg for diabetes, ischemic heart disease, stroke, CKD, or high CVD risk >=30%) is not met at Step 1, advance to Step 2; if not met at Step 2 after 1 month, advance to Step 3 (Amlodipine 5 mg + Losartan 50 mg + Hydrochlorothiazide 12.5 mg once daily, or increase Amlodipine to 10 mg at Step 2 and double Losartan to 100 mg at Step 3); if not met at Step 3 after 1 month, Step 4 refer to a specialist.",
    "medication_details": [
      {
        "name": "Amlodipine (Step 1 or Combination)",
        "dose": "5 mg (2.5 mg starting dose for elderly or low body weight; may increase to 10 mg)",
        "route": "Oral",
        "frequency": "Once daily",
        "duration": "Long-term / life-long maintenance with monthly review until target reached",
        "instructions": "First-line monotherapy for BP 140-159/90-99 mmHg; combine with Losartan 50 mg immediately if initial BP >= 160/100 mmHg."
      },
      {
        "name": "Losartan (Step 2 Combination)",
        "dose": "50 mg (may double to 100 mg in Step 3 alternative)",
        "route": "Oral",
        "frequency": "Once daily",
        "duration": "Long-term maintenance, review after 1 month",
        "instructions": "Combine with Amlodipine 5 mg once daily for Step 2 or as immediate initial therapy if BP >= 160/100 mmHg. Check serum creatinine and potassium."
      },
      {
        "name": "Hydrochlorothiazide (Step 3 Combination)",
        "dose": "12.5 mg",
        "route": "Oral",
        "frequency": "Once daily",
        "duration": "Long-term maintenance if Step 2 fails after 1 month",
        "instructions": "Added to Amlodipine 5 mg + Losartan 50 mg at Step 3 after checking medication adherence."
      }
    ],
    "exclusions_or_missing_info": [
      "STRICT EXCLUSION: DO NOT prescribe Losartan, any ACE inhibitor, ARB, or thiazide diuretic to pregnant patients or women of childbearing age who may become pregnant",
      "Do not use this primary care protocol for pregnant patients or children <18 years old (refer to specialist)",
      "Missing information to check: Sex, age, pregnancy/childbearing status, body weight, comorbidities (diabetes, CKD, CVD) to set BP target (<140/90 vs <130/80 mmHg), and medication adherence"
    ],
    "source": {
      "title": "Bangladesh Guidelines on Hypertension",
      "version": "Second Edition 2023",
      "pdf_page": "64-65",
      "evidence_passage": "Step 1: Start amlodipine 5 mg once daily. Step 2: Review after 1 month. If treatment target not met, amlodipine 5 mg + losartan 50 mg once daily. Step 3: amlodipine 5 mg + losartan 50 mg + hydrochlorothiazide 12.5 mg once daily. Step 4: refer to a specialist. Footnote *: For patients with SBP >=160 mmHg or DBP >=100 mmHg, start immediate treatment at step 2 (amlodipine 5 mg + losartan 50 mg)."
    }
  },
  {
    "card_id": "HTN_MED_02",
    "disease": "Hypertension",
    "clinical_category": "medication_guidance",
    "prerequisites": [
      "Female patient with Hypertension who is of childbearing age (may become pregnant) OR currently pregnant"
    ],
    "recommended_action": "If the patient is CURRENTLY PREGNANT: DO NOT use the primary care hypertension drug protocol; refer immediately to an appropriate specialist. If the patient is a WOMAN OF CHILDBEARING AGE who may become pregnant (and is not currently pregnant): strictly avoid Losartan, ACE inhibitors, ARBs, and hydrochlorothiazide. Follow the dedicated Childbearing Age pathway: Step 1: Start Amlodipine 5 mg once daily. Step 2: Review after 1 month; if target BP not met, increase to Amlodipine 10 mg once daily. Step 3: Review after 1 month; if target not met, refer to a specialist.",
    "medication_details": [
      {
        "name": "Amlodipine (Women of Childbearing Age Pathway)",
        "dose": "Step 1: 5 mg; Step 2 (if target not met at 1 month): 10 mg",
        "route": "Oral",
        "frequency": "Once daily",
        "duration": "Review monthly; refer to specialist at Step 3 if 10 mg fails after 1 month",
        "instructions": "Only for non-pregnant women of childbearing age. If patient is pregnant, do not initiate primary care protocol—refer immediately to a specialist."
      }
    ],
    "exclusions_or_missing_info": [
      "ABSOLUTE CONTRAINDICATION: DO NOT PRESCRIBE Losartan or any ACE inhibitor or ARB or thiazide to women who may become pregnant or who are pregnant",
      "Footnote *** & Section 5.3.6: This protocol must not be used for pregnant patients. If patient is pregnant, refer to a specialist.",
      "Missing information to check: Confirm current pregnancy status (pregnancy test) in any female patient of reproductive age before prescribing antihypertensives"
    ],
    "source": {
      "title": "Bangladesh Guidelines on Hypertension",
      "version": "Second Edition 2023",
      "pdf_page": "64-65",
      "evidence_passage": "For Women of Childbearing Age Who May Become Pregnant: Step 1: Start amlodipine 5 mg once daily. Step 2: Review after 1 month. If treatment target not met, amlodipine 10 mg once daily. Step 3: Review after 1 month. If treatment target not met, refer to specialist. WARNING: DO NOT PRESCRIBE losartan or any ACE inhibitor or ARB or thiazide to women who may become pregnant. Footnote ***: This protocol must not be used for pregnant patients. If patient is pregnant, refer to a specialist."
    }
  },
  {
    "card_id": "HTN_ADV_01",
    "disease": "Hypertension",
    "clinical_category": "advice",
    "prerequisites": [
      "Adult with elevated blood pressure or confirmed Hypertension"
    ],
    "recommended_action": "Provide comprehensive lifestyle modification advice: (1) Avoid all alcohol and tobacco (including second-hand smoke); (2) Physical activity >= 2.5 hours per week; (3) Reduce salt intake to < 1 teaspoon per day (avoid added table salt, crisps, salted nuts, processed foods, soy/fish sauce); (4) Eat a heart-healthy diet (>= 5 servings of vegetables/fruit daily, healthy oils like sunflower/olive/sesame, lean meats/poultry, omega-3 rich fish twice weekly, avoid added sugars/sweets/sugary drinks, lose weight if overweight). Follow up monthly until target BP is achieved (within 3 months), then every 3-6 months. Refer to a specialist if accelerated/malignant HTN, suspected secondary HTN, resistant HTN, recent target organ damage, pregnancy, or age <18 years.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Check medication adherence and financial constraints at every follow-up visit before escalating drug therapy",
      "Immediate specialist referral required for: pregnancy, age <18 years, accelerated/malignant HTN, suspected secondary HTN, resistant HTN, or recent onset HMOD"
    ],
    "source": {
      "title": "Bangladesh Guidelines on Hypertension",
      "version": "Second Edition 2023",
      "pdf_page": "49, 64-65",
      "evidence_passage": "Advice for lifestyle modification: Avoid alcohol and tobacco... Exercise >=2.5 hrs/week; Reduce salt to <1 teaspoon/day; Eat a heart-healthy diet... A monthly follow up after initiation or a change in antihypertensive medications until patients reach target should be done. Once target BP is achieved, follow-up at three to six-month interval is appropriate... Indications for referral: accelerated or malignant hypertension, suspected secondary hypertension, resistant hypertension, recent onset of target organ damage, pregnancy, children <18 years old."
    }
  },
  {
    "card_id": "CHOLERA_DIAG_01",
    "disease": "Cholera",
    "clinical_category": "probable_diagnosis",
    "prerequisites": [
      "Acute watery diarrhoea (>= 3 loose/watery stools in 24 hours) with or without vomiting"
    ],
    "recommended_action": "Assess and classify the patient's dehydration status immediately to assign the treatment plan: (1) No Sign of Dehydration -> assign Plan A (oral ORS at home/outpatient); (2) Some Dehydration -> assign Plan B (monitored 4-hour ORS therapy in facility); (3) Severe Dehydration -> assign Plan C (immediate IV Ringer's Lactate resuscitation). Also check if patient belongs to high-risk group (pregnant women, elderly, or severe/some dehydration requiring antibiotics) or has hypoglycaemia signs.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Missing information to check: Exact dehydration severity signs (mental state: alert vs restless/irritable vs lethargic/unconscious; radial pulse; eyes; skin pinch; thirst), patient age and exact body weight in kg (essential for fluid volume calculation), pregnancy status, and blood glucose level"
    ],
    "source": {
      "title": "WHO Clinical Tools for Cholera Treatment Facilities",
      "version": "2024",
      "pdf_page": "5-9, 12, 14",
      "evidence_passage": "Treatment for Some dehydration: Plan B (ORS)... Treatment for No sign of dehydration: Plan A (ORS)... Cholera fluids quick reference: Plan C (100 mL/kg Ringers in 3 hours for children >1 yr and adults; 6 hours for infants <1 yr)... Anyone with at least 3 episodes of diarrhoea in 24 hours should seek treatment."
    }
  },
  {
    "card_id": "CHOLERA_INV_01",
    "disease": "Cholera",
    "clinical_category": "investigations",
    "prerequisites": [
      "Suspected Cholera / acute watery diarrhoea under clinical assessment or rehydration"
    ],
    "recommended_action": "Perform bedside clinical hydration reassessment (at least every hour during Plan B ORS therapy, and continuously during Plan C IV resuscitation) and check blood glucose to screen for hypoglycaemia (defined as blood glucose < 3.0 mmol/L or < 54 mg/dL). If hypoglycaemia is treated with IV dextrose, recheck blood glucose after 30 minutes.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Missing information to check: Baseline blood glucose measurement (mmol/L or mg/dL), ongoing stool loss frequency, and hourly hydration reassessment status",
      "Do not delay rehydration while performing assessments"
    ],
    "source": {
      "title": "WHO Clinical Tools for Cholera Treatment Facilities",
      "version": "2024",
      "pdf_page": "6, 12",
      "evidence_passage": "Plan B: Reassess at least every hour... Hypoglycaemia quick reference: Glucose less than 3.0 mmol/L (54 mg/dL) -> Give Dextrose bolus -> after 30 mins Recheck glucose."
    }
  },
  {
    "card_id": "CHOLERA_MED_01",
    "disease": "Cholera",
    "clinical_category": "medication_guidance",
    "prerequisites": [
      "Suspected Cholera with No Dehydration (Plan A) OR Some Dehydration (Plan B)"
    ],
    "recommended_action": "For Plan B (Some Dehydration): Administer ORS over the first 4 hours according to age/weight table (<4 mos/<5 kg: 200-400 mL; 4-11 mos/5-8 kg: 400-600 mL; 12-23 mos/8-12 kg: 600-800 mL; 2-4 yrs/12-16 kg: 800-1200 mL; 4-14 yrs/16-30 kg: 1200-2200 mL; 15+ yrs/30+ kg: 2200-4000 mL), plus additional ORS after each loose stool (<24 mos: 50-100 mL; 2-9 yrs: 100-200 mL; 10+ yrs: as much as wanted). For Plan A (No Dehydration): Give ORS after each loose stool at the same age-based volume and provide daily ORS packets (<24 mos: 500 mL/day; 2-9 yrs: 1 L/day; 10+ yrs: 2 L/day). For ALL pediatric patients: give oral Zinc for 10 days (<6 months: 10 mg once daily; >6 months: 20 mg once daily). Note: All pregnant women and elderly patients should receive antibiotics regardless of hydration status or treatment plan.",
    "medication_details": [
      {
        "name": "Oral Rehydration Solution (ORS) - Plan B (Some Dehydration)",
        "dose": "15+ yrs (30+ kg): 2200-4000 mL; 4-14 yrs (16-30 kg): 1200-2200 mL; 2-4 yrs (12-16 kg): 800-1200 mL; 12-23 mos (8-12 kg): 600-800 mL; 4-11 mos (5-8 kg): 400-600 mL; <4 mos (<5 kg): 200-400 mL",
        "route": "Oral",
        "frequency": "Given over 4 hours, plus ongoing loss replacement after each loose stool (<24 mos: 50-100 mL; 2-9 yrs: 100-200 mL; 10+ yrs: as much as wanted)",
        "duration": "Initial 4 hours (reassess hourly), then maintenance until diarrhoea stops",
        "instructions": "Give additional ORS to replace ongoing stool losses in addition to the 4-hour table volume."
      },
      {
        "name": "Zinc Sulfate (Paediatric patients)",
        "dose": "Under 6 months: 10 mg; Over 6 months: 20 mg",
        "route": "Oral",
        "frequency": "Once per day",
        "duration": "10 days",
        "instructions": "Give to paediatric patients; provide sufficient zinc at discharge to complete the full 10-day course."
      },
      {
        "name": "Antibiotics (Per WHO Cholera Protocol indication)",
        "dose": "Regimen per local/WHO antibiotic chart (indicated for severe dehydration, high purging, and all pregnant or elderly patients)",
        "route": "Oral",
        "frequency": "Per protocol antibiotic schedule",
        "duration": "Single dose or 3-day course depending on agent",
        "instructions": "All pregnant women and elderly patients should receive antibiotics regardless of their hydration status or treatment plan."
      }
    ],
    "exclusions_or_missing_info": [
      "If Severe Dehydration is present, do not rely on oral Plan A/B alone—escalate immediately to Plan C IV Ringer's Lactate",
      "Missing information to check: Patient age and exact weight in kg (to select exact ORS volume bracket), dehydration grade, pregnancy status, and blood glucose"
    ],
    "source": {
      "title": "WHO Clinical Tools for Cholera Treatment Facilities",
      "version": "2024",
      "pdf_page": "6, 8, 14",
      "evidence_passage": "Plan B (ORS): 1 Give ORS over 4 hours (15+ years / 30+ kg: 2200-4000 mL)... 2 Reassess at least every hour. 3 Give additional ORS to replace ongoing losses... Zinc - give for 10 days: In patients under 6 months, 10mg once per day; In patients over 6 months, 20mg once per day. Antibiotics: All pregnant women and elderly patients should receive antibiotics regardless of their hydration status or treatment plan."
    }
  },
  {
    "card_id": "CHOLERA_MED_02",
    "disease": "Cholera",
    "clinical_category": "medication_guidance",
    "prerequisites": [
      "Suspected Cholera with Severe Dehydration (Plan C) OR Hypoglycaemia (Blood glucose < 3.0 mmol/L or < 54 mg/dL)"
    ],
    "recommended_action": "For Plan C (Severe Dehydration): Administer IV Ringer's Lactate totaling 100 mL/kg: in children >1 year and adults, give over 3 hours (First bolus: 30 mL/kg over 30 mins; followed by 70 mL/kg over 2.5 hours, e.g., for >=55 kg: 6000 mL total = 2000 mL in 30 mins + 4000 mL in 2.5 hrs). In infants <1 year, give 100 mL/kg over 6 hours (30 mL/kg over 1 hour, then 70 mL/kg over 5 hours). For Hypoglycaemia (Glucose < 3.0 mmol/L / 54 mg/dL): In adults, give IV Dextrose 50% (25-50 mL) OR Dextrose 10% (125-250 mL) over 2 minutes; in children, give IV Dextrose 10% (5 mL/kg) over 2 minutes. Recheck glucose after 30 mins, repeat 2nd bolus if necessary, and start maintenance dextrose infusion (800 mL Ringer's Lactate + 200 mL 50% Dextrose at 50 mL/hr in adults or 5 mL/kg/hr in children) if recurrent.",
    "medication_details": [
      {
        "name": "Ringer's Lactate IV (Plan C Severe Dehydration)",
        "dose": "100 mL/kg total (30 mL/kg initial bolus + 70 mL/kg continuation; e.g., 50-54 kg: 1500 mL in 30 mins + 3500 mL in 2.5 hrs; >=55 kg: 2000 mL in 30 mins + 4000 mL in 2.5 hrs)",
        "route": "Intravenous (IV)",
        "frequency": "Continuous rapid infusion",
        "duration": "3 hours total for children >1 yr & adults (30 mins bolus + 2.5 hrs); 6 hours total for infants <1 yr (1 hr bolus + 5 hrs)",
        "instructions": "Administer through an appropriate infusion set."
      },
      {
        "name": "Dextrose IV (For Hypoglycaemia < 3.0 mmol/L or < 54 mg/dL)",
        "dose": "Adults: 25-50 mL of Dextrose 50% OR 125-250 mL of Dextrose 10%. Children: 5 mL/kg of Dextrose 10%",
        "route": "Intravenous (IV)",
        "frequency": "Bolus over 2 minutes; recheck glucose at 30 mins and give 2nd bolus if necessary",
        "duration": "Immediate bolus; if recurrent, maintenance infusion of 800 mL RL + 200 mL 50% dextrose at 50 mL/hr (adults) or 5 mL/kg/hr (children)",
        "instructions": "If no IV access, 2-5 mL of 50% dextrose fluid can be placed inside the cheek using a syringe (do not use needle or spoon)."
      }
    ],
    "exclusions_or_missing_info": [
      "Missing information to check: Exact body weight in kg and age (<1 year vs >1 year) to select the 6-hour infant vs 3-hour adult/child IV schedule, and blood glucose level"
    ],
    "source": {
      "title": "WHO Clinical Tools for Cholera Treatment Facilities",
      "version": "2024",
      "pdf_page": "9, 12",
      "evidence_passage": "Cholera fluids quick reference: Children >1 year old, and adults (plan C is 100mL/kg in 3 hours): 30 mL/kg bolus Ringers over 30 mins, 70 mL/kg Ringers over 2.5 hours. Infants (under 1 year, plan C 100mL/kg in 6 hours): 30 mL/kg Ringers over 1 hour, 70 mL/kg Ringers over 5 hours... Hypoglycaemia quick reference: Glucose less than 3.0 mmol/L (54 mg/dL) -> Adults: Dextrose 50% 25-50 mL or Dextrose 10% 125-250 mL IV over 2 minutes; Children: Dextrose 10% 5 mL/kg IV over 2 minutes."
    }
  },
  {
    "card_id": "CHOLERA_ADV_01",
    "disease": "Cholera",
    "clinical_category": "advice",
    "prerequisites": [
      "Patient recovering from Cholera / acute watery diarrhoea being prepared for discharge or home care"
    ],
    "recommended_action": "Before discharge: (1) Provide at least 4 packets of ORS for all discharged patients to use at home (which may also be used for family members or neighbours who develop diarrhoea); (2) For paediatric patients, provide sufficient Zinc tablets to complete the full 10-day treatment; (3) Counsel patients and caregivers on cholera prevention (safe water, handwashing, sanitation); (4) Instruct them that anyone with >= 3 episodes of diarrhoea in 24 hours must seek medical treatment immediately.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Verify patient can tolerate oral fluids and has no signs of dehydration before discharge"
    ],
    "source": {
      "title": "WHO Clinical Tools for Cholera Treatment Facilities",
      "version": "2024",
      "pdf_page": "14",
      "evidence_passage": "Discharge assessment: Ensure patients and their caregivers understand that anyone with at least 3 episodes of diarrhoea in 24 hours should seek treatment. Ensure all patients and their caregivers understand how to prevent cholera before discharge. Provide at least 4 packets of ORS for all patients who are discharged to use at home... For paediatric patients give sufficient zinc to complete 10 days treatment."
    }
  },
  {
    "card_id": "CHIK_DIAG_01",
    "disease": "Chikungunya",
    "clinical_category": "probable_diagnosis",
    "prerequisites": [
      "Sudden/acute onset of high fever (> 38.5 C / 101.3 F, typically lasting 24-48 hours)",
      "Severe arthralgia or arthritis (typically symmetric, involving ankles, wrists, and small joints of hands, worse in the morning, relieved by mild exercise, or causing a stooped/bending posture) and/or maculopapular rash"
    ],
    "recommended_action": "Categorize diagnosis as: Possible Case (meets clinical criteria of acute fever >38.5 C + severe arthralgia/arthritis), Probable Case (meets clinical criteria + epidemiological criteria of residing in/visiting an epidemic area within 15 days), or Confirmed Case (laboratory confirmed). Classify clinical severity into Mild (low-grade fever, mild arthralgia/myalgia, rash), Moderate (persistent fever, moderate arthralgia/arthritis, generalized myalgia, hypotension, mild bleeding, retro-orbital pain, oliguria), or Severe (persistent high fever, severe arthritis, persistent vomiting/diarrhoea, altered sensorium, GI bleeding, shock). Mild/moderate cases without high-risk features can be managed at home; severe or high-risk cases require hospital admission.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Check Hospital Admission Criteria (Page 19): hemodynamic instability (syncope, SBP <90 mmHg, pulse pressure <30 mmHg), oliguria (<500 mL/24 hrs), altered sensorium, bleeding, persistent disabling joint pain after 3 days of treatment, age >60 years, infants <1 year, pregnancy, or high-risk comorbidities (HTN, diabetes, CAD/CVD, COPD, co-infection with Dengue/TB/Malaria)",
      "Manage as Dengue until Dengue is ruled out"
    ],
    "source": {
      "title": "National Guideline on Chikungunya Management",
      "version": "2017",
      "pdf_page": "11-15, 18-19",
      "evidence_passage": "Chikungunya Fever should be suspected when a person develops sudden onset of fever, joint manifestations and rash... Clinical criteria: acute onset of fever >38.5 C, severe arthralgia/arthritis not explained by other medical conditions. Epidemiological criteria: Residing or having visited epidemic areas... Involvement is symmetric and often ankles, wrists and small joints of the hand are the worst affected."
    }
  },
  {
    "card_id": "CHIK_INV_01",
    "disease": "Chikungunya",
    "clinical_category": "investigations",
    "prerequisites": [
      "Suspected or probable Chikungunya fever"
    ],
    "recommended_action": "Order routine laboratory investigations: CBC (typically shows leucopenia/lymphopenia; thrombocytopenia is rare unlike Dengue), ESR (usually elevated), C-Reactive Protein (CRP, increased in acute phase), and SGPT (elevated). For specific confirmation depending on illness duration: within 5 days of illness onset, order Real-Time RT-PCR for Chikungunya viral RNA (performed at IEDCR) or virus isolation; between 5 to 28 days of fever onset, order Chikungunya-specific IgM antibody test (or paired IgG 3 weeks apart). Simultaneously test for Dengue (e.g., Dengue NS1 antigen if <=5 days) to exclude Dengue.",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Note: Diagnostic validity of currently available rapid ICT kits for Chikungunya IgM/IgG is very poor",
      "If neurological features (meningo-encephalitis) are present, CSF specimen is indicated",
      "Missing information to check: Exact duration of fever in days (to choose RT-PCR [<=5 days] vs IgM serology [5-28 days]) and Dengue NS1 status"
    ],
    "source": {
      "title": "National Guideline on Chikungunya Management",
      "version": "2017",
      "pdf_page": "11, 21-22, 26",
      "evidence_passage": "Routine Laboratory Investigations: CBC (Leucopenia; Thrombocytopenia: rare), ESR: Usually Elevated, C-Reactive Protein: Increased during the Acute Phase, SGPT: Elevated. Specific Laboratory Investigations: Presence of viral RNA by real Time RT-PCR (Within 5 days of onset of Illness); Presence of viral specific IgM antibody in single serum sample collected within 5 to 28 days of onset Fever."
    }
  },
  {
    "card_id": "CHIK_MED_01",
    "disease": "Chikungunya",
    "clinical_category": "medication_guidance",
    "prerequisites": [
      "Suspected or confirmed Chikungunya fever in acute stage (first 7-10 days)"
    ],
    "recommended_action": "Prescribe Paracetamol for fever and joint pain, oral rehydration fluids with salt/electrolytes (~2 litres/24 hours), and oral antihistamines if skin itching/pruritus is present. Strictly avoid aspirin, self-medicated NSAIDs, and systemic steroids during the acute stage (first 7-10 days). After 7-10 days (post-acute phase), a physician may prescribe an NSAID ONLY IF Dengue is excluded (Dengue NS1 negative, afebrile for at least 48 hours, and no dengue warning signs). For refractory chronic arthralgia, hydroxychloroquine 200 mg orally once daily (or chloroquine phosphate 300 mg/day) for 4 weeks may be used.",
    "medication_details": [
      {
        "name": "Paracetamol",
        "dose": "Adults: up to 1000 mg per dose (up to 4 times daily, max 4000 mg/day if no liver/kidney disease); Children: 50-60 mg/kg/day in divided doses",
        "route": "Oral",
        "frequency": "Up to 4 times daily during periods of fever/pain",
        "duration": "Acute febrile and pain phase",
        "instructions": "Use only in persons with no pre-existing liver or kidney disease. Note: If Dengue has not yet been excluded, adhere to the stricter Dengue Paracetamol cap of max 3 gm/day in adults."
      },
      {
        "name": "Oral Fluids with Electrolytes",
        "dose": "Approximately 2 litres per 24 hours",
        "route": "Oral",
        "frequency": "Throughout the day",
        "duration": "During acute illness",
        "instructions": "Consume plenty of water with electrolytes / home-available fluids with salt."
      },
      {
        "name": "Antihistamines (if itching/pruritus present)",
        "dose": "Standard symptomatic dose",
        "route": "Oral",
        "frequency": "As needed for itching",
        "duration": "During acute rash/itching phase",
        "instructions": "Can be used for pruritic skin rash; Zinc oxide cream or Calamine lotion may be used for hyperpigmentation/papular eruptions."
      }
    ],
    "exclusions_or_missing_info": [
      "STRICT EXCLUSION: Avoid Aspirin and NSAIDs during the first 10 days and until Dengue is adequately excluded (NS1 negative, afebrile >=48 hrs, no warning signs). Steroids have NO role in the acute stage.",
      "Paracetamol full dose requires excluding pre-existing liver or kidney disease",
      "Missing information to check: Pre-existing liver/kidney disease, duration of illness (<=10 days vs >10 days), Dengue exclusion status, and hospital admission criteria (pregnancy, age >60 or <1 yr, hemodynamic instability)"
    ],
    "source": {
      "title": "National Guideline on Chikungunya Management",
      "version": "2017",
      "pdf_page": "18-20, 26",
      "evidence_passage": "Consume plenty of water with electrolytes (approximately 2 litres of home available fluids with salt in 24 hours). Take paracetamol tablets during periods of fever (up to 1000 mg tablets four times daily), in persons with no preexisting liver or kidney disease. Children may be given 50-60 mg per kg body weight per day in divided doses... Avoid self medication with aspirin or NSAIDs. Antihistamines can be used for itching... Aspirin or NSAIDs should be avoided during first 10 days. NSAIDs can be used in Chikungunya only when Dengue fever is adequately excluded. Steroid has no role in acute stage."
    }
  },
  {
    "card_id": "CHIK_ADV_01",
    "disease": "Chikungunya",
    "clinical_category": "advice",
    "prerequisites": [
      "Suspected or confirmed Chikungunya fever"
    ],
    "recommended_action": "In the ACUTE phase (first 7-10 days): Advise adequate rest in a warm environment, refrain from exertion, and apply COLD compresses to painful joints to reduce joint damage. Strictly avoid heat application (worsens joint pain) and DO NOT perform formal physiotherapy in the acute phase (maintain only normal gentle joint/body movement). In the POST-ACUTE / RECOVERING phase (after 7-10 days): Recommend mild exercises (walking on level ground, active hand movements, proper joint posturing to prevent stiffness) and refer patients with persistent arthralgia/arthritis or loss of autonomy to an arthralgia clinic / physical medicine specialist for rehabilitation. Prevent mosquito transmission (daytime-biting Aedes control, mosquito nets/repellents). Nursing mothers are encouraged to continue breastfeeding (not transmitted via breast milk).",
    "medication_details": [],
    "exclusions_or_missing_info": [
      "Do not advise heat compresses or physiotherapy during the acute stage",
      "Admit immediately if patient is pregnant (risk of perinatal vertical transmission), >60 years, <1 year, oliguric, bleeding, hypotensive, or has persistent disabling arthritis after 3 days of symptomatic treatment",
      "Consult ophthalmologist if ocular symptoms (uveitis/blurred vision) occur"
    ],
    "source": {
      "title": "National Guideline on Chikungunya Management",
      "version": "2017",
      "pdf_page": "18-22, 26-27",
      "evidence_passage": "Cold compresses may help in reducing joint damage. Heat may increase/worsen joint pain and is therefore best to avoid during acute stage. Refrain from exertion... Opinion of Expert Committee (Page 26): The experts agreed that there is no indication of physiotherapy in acute phase. Only cold compression can be advised. Mild forms of exercise are recommended in post-acute phase (walking on level grounds, active hand movements, proper posturing)."
    }
  }
]

with open("canonical_cards.json", "w", encoding="utf-8") as f:
    json.dump(canonical_cards, f, indent=2, ensure_ascii=False)

print(f"Successfully saved {len(canonical_cards)} canonical cards to canonical_cards.json!")

Successfully saved 19 canonical cards to canonical_cards.json!


In [3]:
#Local Model+TF-IDF RAG+Gradio UI
!pip install -q gradio scikit-learn transformers accelerate torch

import os
import json
import time
import numpy as np
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading local model {MODEL_NAME}... (takes ~30 seconds the first time)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded successfully!")

def generate_local_llm(prompt):
    messages = [
        {"role": "system", "content": "You are a clinical decision support engine grounded strictly in the provided JSON protocol cards."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=650,
            temperature=0.1,
            do_sample=False
        )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


with open("canonical_cards.json", "r", encoding="utf-8") as f:
    CARDS = json.load(f)

def card_to_search_text(card):
    prereqs = " ".join(card.get("prerequisites", []))
    exclusions = " ".join(card.get("exclusions_or_missing_info", []))
    return (
        f"{card['disease']} {card['clinical_category']} "
        f"{prereqs} {card['recommended_action']} {exclusions} "
        f"{card['source']['evidence_passage']}"
    )

CORPUS_TEXTS = [card_to_search_text(c) for c in CARDS]
VECTORIZER = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
CARD_VECTORS = VECTORIZER.fit_transform(CORPUS_TEXTS)


def retrieve_cards(patient_query, mode="Baseline"):
    query_vec = VECTORIZER.transform([patient_query])
    scores = cosine_similarity(query_vec, CARD_VECTORS).flatten()

    if mode == "Baseline":
        top_indices = np.argsort(scores)[::-1][:4]
    else:
        selected_indices = []
        categories = ["probable_diagnosis", "investigations", "medication_guidance", "advice"]
        for cat in categories:
            cat_indices = [i for i, c in enumerate(CARDS) if c["clinical_category"] == cat]
            best_in_cat = max(cat_indices, key=lambda idx: scores[idx])
            selected_indices.append(best_in_cat)

        for idx in np.argsort(scores)[::-1]:
            if idx not in selected_indices and len(selected_indices) < 6:
                selected_indices.append(idx)
        top_indices = selected_indices

    retrieved = []
    for idx in top_indices:
        c = CARDS[idx].copy()
        c["retrieval_score"] = round(float(scores[idx]), 4)
        retrieved.append(c)
    return retrieved


DEMO_CASES = {
    "Custom / Live Interview Case": {
        "complaints": "", "duration": "", "age": "", "sex": "Male",
        "weight": "", "pregnancy": "Not Applicable", "vitals": "",
        "history": "", "medications": "", "allergies": ""
    },
    "Case 1: Ordinary Dengue (Day 3 Febrile)": {
        "complaints": "Sudden onset high continuous fever (39.5 C), severe headache, retro-orbital pain, muscle and joint pain (myalgia/arthralgia), mild nausea. No abdominal pain, no bleeding, no persistent vomiting.",
        "duration": "3 days",
        "age": "28", "sex": "Male", "weight": "62 kg",
        "pregnancy": "Not Applicable",
        "vitals": "BP 118/76 mmHg, Pulse 88 bpm, normal urine output",
        "history": "No diabetes, no hypertension, no kidney or liver disease",
        "medications": "None", "allergies": "None known"
    },
    "Case 2: Ordinary Hypertension (Grade 2 Primary Care)": {
        "complaints": "Asymptomatic; high blood pressure detected during routine clinic visit. No chest pain, no breathlessness, no visual changes or dizziness.",
        "duration": "Detected today on initial visit",
        "age": "48", "sex": "Male", "weight": "72 kg",
        "pregnancy": "Not Applicable",
        "vitals": "Office BP 165/102 mmHg (confirmed across 3 readings in both arms), Pulse 76 bpm",
        "history": "Sedentary lifestyle, high dietary salt",
        "medications": "None", "allergies": "None known"
    },
    "Case 3: Ordinary Cholera (Some Dehydration - Plan B)": {
        "complaints": "Acute profuse watery diarrhoea (8 episodes in the last 24 hours), thirsty, restless and irritable. Able to drink oral fluids.",
        "duration": "1 day (24 hours)",
        "age": "25", "sex": "Male", "weight": "55 kg",
        "pregnancy": "Not Applicable",
        "vitals": "BP 110/70 mmHg, eyes slightly sunken, blood glucose 4.5 mmol/L (81 mg/dL)",
        "history": "Previously healthy adult",
        "medications": "None", "allergies": "None known"
    },
    "Case 4: Ordinary Chikungunya (Acute Symmetric Arthritis)": {
        "complaints": "Sudden high fever (39.0 C) with severe symmetric joint pain and swelling in wrists, ankles, and small hand joints, worse in the morning, stooped posture.",
        "duration": "2 days",
        "age": "35", "sex": "Male", "weight": "68 kg",
        "pregnancy": "Not Applicable",
        "vitals": "BP 120/80 mmHg, normal urine output, alert",
        "history": "Resides in Dhaka area with active chikungunya transmission; no liver or kidney disease",
        "medications": "None", "allergies": "None known"
    },
    "Case 5: Ambiguous / Incomplete Febrile Case": {
        "complaints": "Fever, headache, generalized body and joint ache, and mild skin rash.",
        "duration": "Unknown (Not stated)",
        "age": "30", "sex": "Female", "weight": "Unknown",
        "pregnancy": "Unknown (Not asked)",
        "vitals": "Not recorded",
        "history": "Unknown",
        "medications": "Unknown", "allergies": "Unknown"
    },
    "Case 6: Modified Case 2 (Hypertension in Pregnant / Childbearing Female)": {
        "complaints": "Asymptomatic elevated blood pressure found at primary health care checkup. No headache, visual disturbance, or chest pain.",
        "duration": "Detected today",
        "age": "29", "sex": "Female", "weight": "60 kg",
        "pregnancy": "Currently 8 weeks pregnant (Woman of childbearing age)",
        "vitals": "Office BP 165/102 mmHg (repeated in both arms)",
        "history": "First pregnancy, no chronic kidney disease",
        "medications": "Prenatal folic acid", "allergies": "None known"
    }
}


def run_recommendation_engine(complaints, duration, age, sex, weight, pregnancy, vitals, history, medications, allergies, mode):
    start_time = time.time()

    patient_summary = (
        f"- Complaints: {complaints or 'Not provided'}\n"
        f"- Duration: {duration or 'Not provided'}\n"
        f"- Demographics: Age {age or '?'}, Sex {sex or '?'}, Weight {weight or '?'}\n"
        f"- Pregnancy Status: {pregnancy or 'Not provided'}\n"
        f"- Vitals: {vitals or 'Not provided'}\n"
        f"- History: {history or 'Not provided'}\n"
        f"- Medications: {medications or 'None'}\n"
        f"- Allergies: {allergies or 'None'}"
    )

    retrieved_cards = retrieve_cards(patient_summary, mode=mode)

    evidence_md = f"### Retrieved Canonical Cards ({mode} Mode — {len(retrieved_cards)} Cards)\n"
    compact_cards = []
    for c in retrieved_cards:
        evidence_md += (
            f"- **`{c['card_id']}`** ({c['disease']} | `{c['clinical_category']}` | Score: `{c['retrieval_score']}`)\n"
            f"  - *Source:* {c['source']['title']}, **PDF Page: {c['source']['pdf_page']}**\n"
        )
        compact_cards.append({
            "card_id": c["card_id"],
            "disease": c["disease"],
            "category": c["clinical_category"],
            "prerequisites": c.get("prerequisites", []),
            "action": c["recommended_action"],
            "medication_details": c.get("medication_details", []),
            "exclusions_or_missing_info": c.get("exclusions_or_missing_info", []),
            "citation": f"{c['source']['title']}, Page {c['source']['pdf_page']}"
        })

    extra_rule = ""
    if mode == "Improved":
        extra_rule = "\nIMPROVED MODE RULE: Explicitly verify prerequisites and exclusions (especially pregnancy/childbearing age in Hypertension and unexcluded Dengue in Chikungunya) before outputting Medication Planning."

    prompt = f"""Use ONLY the retrieved cards below. Do NOT guess missing details.{extra_rule}

Write your response using these exact 5 headings:
## 1. Missing Information & Applicability Check
## 2. Probable Diagnosis (cite [Card ID | Page])
## 3. Investigations (cite [Card ID | Page])
## 4. Medication Planning (include Name, Dose, Route, Frequency, Duration, Instructions & Contraindications; cite [Card ID | Page])
## 5. Advice & Follow-Up (cite [Card ID | Page])

PATIENT:
{patient_summary}

RETRIEVED CARDS:
{json.dumps(compact_cards, indent=1)}
"""

    llm_text = generate_local_llm(prompt)
    elapsed = round(time.time() - start_time, 2)
    header = f"**Mode:** `{mode}` | **Model:** `{MODEL_NAME} (Local)` | **Response Time:** `{elapsed}s`\n\n---\n"
    return patient_summary, evidence_md, header + llm_text

def load_preset_case(case_name):
    c = DEMO_CASES[case_name]
    return (c["complaints"], c["duration"], c["age"], c["sex"], c["weight"],
            c["pregnancy"], c["vitals"], c["history"], c["medications"], c["allergies"])

# 6. Interactive Gradio UI
with gr.Blocks(title="Clinical NLP Recommendation Prototype") as demo:
    gr.Markdown("# Clinical Protocol RAG Recommendation Engine\n*Grounded in Bangladesh Dengue, Hypertension, Chikungunya & WHO Cholera Guidelines*")

    with gr.Row():
        case_selector = gr.Dropdown(
            choices=list(DEMO_CASES.keys()),
            value="Case 1: Ordinary Dengue (Day 3 Febrile)",
            label="Demonstration Cases (or choose Custom to enter a new case)",
            scale=3
        )
        mode_selector = gr.Radio(
            choices=["Baseline", "Improved"],
            value="Improved",
            label="Engine Mode (Experiment Switch)",
            scale=1
        )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Patient Inputs (Disease Name Not Required)")
            complaints_in = gr.Textbox(label="Chief Complaints & Symptoms", lines=3, value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["complaints"])
            duration_in = gr.Textbox(label="Illness Duration", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["duration"])
            with gr.Row():
                age_in = gr.Textbox(label="Age", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["age"])
                sex_in = gr.Dropdown(choices=["Male", "Female", "Other / Unknown"], label="Sex", value="Male")
                weight_in = gr.Textbox(label="Body Weight", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["weight"])
            pregnancy_in = gr.Textbox(label="Pregnancy / Childbearing Status", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["pregnancy"])
            vitals_in = gr.Textbox(label="Vital Signs & Findings", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["vitals"])
            history_in = gr.Textbox(label="Medical History", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["history"])
            meds_in = gr.Textbox(label="Current Medications", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["medications"])
            allergies_in = gr.Textbox(label="Allergies", value=DEMO_CASES["Case 1: Ordinary Dengue (Day 3 Febrile)"]["allergies"])
            submit_btn = gr.Button("Generate Clinical Recommendations", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Recommendation Procedure & Outputs")
            with gr.Accordion("Step 1 & 2: Active Patient Context & Retrieved Evidence", open=True):
                patient_context_out = gr.Textbox(label="Active Patient Context", lines=4, interactive=False)
                retrieved_evidence_out = gr.Markdown(label="Retrieved Canonical Cards")
            recommendations_out = gr.Markdown(label="Clinical Recommendations")

    case_selector.change(
        fn=load_preset_case,
        inputs=[case_selector],
        outputs=[complaints_in, duration_in, age_in, sex_in, weight_in, pregnancy_in, vitals_in, history_in, meds_in, allergies_in]
    )

    submit_btn.click(
        fn=run_recommendation_engine,
        inputs=[complaints_in, duration_in, age_in, sex_in, weight_in, pregnancy_in, vitals_in, history_in, meds_in, allergies_in, mode_selector],
        outputs=[patient_context_out, retrieved_evidence_out, recommendations_out]
    )

demo.launch(share=True, inline=True)

Loading local model Qwen/Qwen2.5-1.5B-Instruct... (takes ~30 seconds the first time)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1970709aa496287a8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [4]:
#Save All
import time
import json

all_results = []
case_names = [k for k in DEMO_CASES.keys() if k != "Custom / Live Interview Case"]

for case_name in case_names:
    c = DEMO_CASES[case_name]
    for mode in ["Baseline", "Improved"]:
        print(f"Running {case_name} [{mode}]...", end=" ", flush=True)
        t0 = time.time()
        p_ctx, ret_md, rec_out = run_recommendation_engine(
            c["complaints"], c["duration"], c["age"], c["sex"], c["weight"],
            c["pregnancy"], c["vitals"], c["history"], c["medications"], c["allergies"],
            mode=mode
        )
        elapsed = round(time.time() - t0, 2)
        print(f"Done ({elapsed}s)")
        all_results.append({
            "case_id": case_name,
            "mode": mode,
            "elapsed_seconds": elapsed,
            "input": c,
            "retrieved_cards_summary": ret_md,
            "recommendation_output": rec_out
        })

with open("six_cases_and_12_outputs.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

with open("six_cases_and_12_outputs.md", "w", encoding="utf-8") as f:
    f.write("# Demonstration: Six Fictional Cases & 12 Saved Engine Outputs\n\n")
    for item in all_results:
        f.write(f"## {item['case_id']} — Mode: {item['mode']} ({item['elapsed_seconds']}s)\n\n")
        f.write(f"{item['retrieved_cards_summary']}\n\n")
        f.write(f"{item['recommendation_output']}\n\n---\n\n")

print("\nALL 12 OUTPUTS SAVED SUCCESSFULLY!")

Running Case 1: Ordinary Dengue (Day 3 Febrile) [Baseline]... Done (640.56s)
Running Case 1: Ordinary Dengue (Day 3 Febrile) [Improved]... Done (840.3s)
Running Case 2: Ordinary Hypertension (Grade 2 Primary Care) [Baseline]... Done (740.54s)
Running Case 2: Ordinary Hypertension (Grade 2 Primary Care) [Improved]... Done (1022.58s)
Running Case 3: Ordinary Cholera (Some Dehydration - Plan B) [Baseline]... Done (589.25s)
Running Case 3: Ordinary Cholera (Some Dehydration - Plan B) [Improved]... Done (969.56s)
Running Case 4: Ordinary Chikungunya (Acute Symmetric Arthritis) [Baseline]... Done (640.95s)
Running Case 4: Ordinary Chikungunya (Acute Symmetric Arthritis) [Improved]... Done (854.46s)
Running Case 5: Ambiguous / Incomplete Febrile Case [Baseline]... Done (564.65s)
Running Case 5: Ambiguous / Incomplete Febrile Case [Improved]... Done (811.18s)
Running Case 6: Modified Case 2 (Hypertension in Pregnant / Childbearing Female) [Baseline]... Done (790.9s)
Running Case 6: Modified Ca